# L10 – Lösningar

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from dlkit.seed import set_seed
from dlkit.train import fit

device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"

def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )

### Övning 1
- a) (32 − 5 + 0)/1 + 1 = 28, så **(16, 28, 28)**.
- b) ⌊(32 − 3 + 2)/2⌋ + 1 = 15 + 1 = 16, så **(16, 16, 16)**.
- c) Faltningarna behåller 28×28 och pooling halverar till 14, så **(8, 14, 14)**.

In [ ]:
answer_a, answer_b, answer_c = (16, 28, 28), (16, 16, 16), (8, 14, 14)
x = torch.randn(1, 3, 32, 32)
assert tuple(nn.Conv2d(3, 16, 5)(x).shape[1:]) == answer_a
assert tuple(nn.Conv2d(3, 16, 3, padding=1, stride=2)(x).shape[1:]) == answer_b
assert tuple(conv_block(3, 8)(torch.randn(1, 3, 28, 28)).shape[1:]) == answer_c
print("Rätt!")

### Övning 2
128 filter, var och ett med 64 × 3 × 3 vikter, plus 128 bias: 128 · 576 + 128 = **73 856**.

In [ ]:
my_count = 128 * 64 * 3 * 3 + 128
assert my_count == sum(p.numel() for p in nn.Conv2d(64, 128, 3).parameters())
print("Rätt!", my_count)

### Övning 3

In [ ]:
raw = datasets.CIFAR10(DATA_DIR, train=True, download=True)
perm = np.random.default_rng(42).permutation(len(raw))
train_idx, val_idx = perm[:45_000], perm[45_000:]
px = raw.data[train_idx] / 255.0
MEAN, STD = tuple(px.mean(axis=(0, 1, 2))), tuple(px.std(axis=(0, 1, 2)))

eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_loader = DataLoader(Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=train_tf), train_idx),
                          batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(Subset(datasets.CIFAR10(DATA_DIR, train=True, transform=eval_tf), val_idx),
                        batch_size=256, num_workers=4)

class WiderCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 64), conv_block(64, 128), conv_block(128, 256))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(256 * 4 * 4, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

set_seed(42)
wide = WiderCNN().to(device)
print(f"{sum(p.numel() for p in wide.parameters()):,} parametrar")
h_wide = fit(wide, train_loader, val_loader, nn.CrossEntropyLoss(),
             torch.optim.AdamW(wide.parameters(), lr=1e-3, weight_decay=0.01), device, epochs=15)
print("bästa val acc:", max(h_wide["val_acc"]))

Den bredare modellen har ungefär 4 gånger fler parametrar och blir oftast några procentenheter bättre, men varje epok tar längre tid. Om vinsten är värd det beror på hur viktig den sista procenten är för ditt problem. Det är ett avvägande, inte ett rätt svar.

### Övning 4 – förslag på svar
1. **Siffror:** små förflyttningar, lite rotation (±10°) och lite skalning är lämpliga. *Olämpligt:* spegling (en spegelvänd 3 är ingen siffra) och stor rotation (6 blir 9).
2. **Röntgen:** lite förflyttning, rotation och kontrast är lämpliga. *Olämpligt:* vänster–höger-spegling kan vara fel, eftersom hjärtat sitter till vänster och spegling skapar anatomiskt omöjliga bilder. Färgförändringar är meningslösa i gråskala.
3. **Satellit:** alla rotationer (90°, 180°) och speglingar är lämpliga, eftersom det inte finns något "upp" sett ovanifrån. Ljus och kontrast är lämpliga för olika tider och väder.